# DKD Risk Prediction in Type 2 Diabetes
## 06. Reviewer Sensitivity Analyses

This notebook addresses reviewer-requested sensitivity analyses that are distinct from
the primary performance evaluation.

The primary one-year corrected cohort, model feature definitions, and
train/calibration/test assignments are treated as frozen. Unless otherwise stated,
sensitivity models are refitted using the primary selected XGBoost hyperparameters
rather than re-optimizing them. This isolates the effect of the sensitivity definition
from a new round of model selection.

Analyses:

1. Pre-landmark renal-abnormality / predictor-outcome circularity sensitivity.
2. Component-specific and diagnosis-code-only DKD outcomes.
3. Longer control-follow-up sensitivity.
4. Fixed 3-year and 5-year post-landmark horizons, with an additional confirmed-survival
   sensitivity when death data are available.
5. Missingness, visit frequency, measurement density, and post-landmark kidney-monitoring
   summaries.
6. Model performance across pre-landmark observation-density strata.
7. PRS-free nested models.
8. Included-versus-excluded cohort characteristics.
9. Independent feature-domain models at the one-year landmark.
10. M1 discrimination versus random performance and PR-AUC prevalence baseline.


# 1. Setup


In [ ]:
import json
import math
import re
import subprocess
import warnings

import gcsfs
import numpy as np
import optuna
import pandas as pd
import xgboost as xgb

from google.cloud import bigquery
from scipy import stats
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
)

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

migration_folder = "rw-migration-aou-rw-639fade3"
bucket = migration_folder

DATA_CURATED_DIR = f"{bucket}/aou_dkd/dkd_data_curation_20260821"
COHORT_DIR = f"{bucket}/aou_dkd/dkd_cohort_20260821"
MODEL_DIR = f"{bucket}/aou_dkd/dkd_models_20260822"
RESULT_DIR = f"{bucket}/aou_dkd/dkd_model_results_20260822"
SENS_DIR = f"{RESULT_DIR}/reviewer_sensitivity_20260822"
SENS_MODEL_DIR = f"{MODEL_DIR}/reviewer_sensitivity_20260822"

fs = gcsfs.GCSFileSystem()

ID_COL = "person_id"
PRIMARY_Y = "dkd_binary"
FINAL_DATE = "20260822"
LANDMARK = "1y"
LANDMARK_DAYS = 365

SPLIT_SEED = 42
TARGET_SPECIFICITY = 0.90
N_BOOT = 1000

MODEL_ORDER = ["M1", "M2", "M3", "M4", "M5"]

# Sensitivity refits use the primary hyperparameters and therefore need only one fit.
XGB_FINAL_N_JOBS = 64

# Independent feature-domain models are separately tuned.
DOMAIN_N_TRIALS = 20
DOMAIN_CV_FOLDS = 5
DOMAIN_CV_N_JOBS = 5
DOMAIN_XGB_N_JOBS = 16

XGB_BOUNDS = {
    "gamma": (1e-8, 1.0),
    "max_depth": (3, 6),
    "min_child_weight": (1, 6),
    "subsample": (0.6, 1.0),
    "colsample_bytree": (0.6, 1.0),
    "learning_rate": (0.01, 0.15),
    "n_estimators": (150, 300),
    "reg_alpha": (1e-8, 0.5),
    "reg_lambda": (1e-8, 0.5),
}


## Resolve the attached All of Us CDR


In [ ]:
def resolve_aou_cdr():
    result = subprocess.run(
        [
            "wb",
            "resource",
            "list",
            "--type=BQ_DATASET",
            "--format=JSON",
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    resources = json.loads(result.stdout)
    candidates = []

    for resource in resources:
        dataset_id = str(
            resource.get("datasetId", "")
        )

        if re.fullmatch(
            r"C\d{4}Q\d+R\d+",
            dataset_id,
        ):
            project_id = resource.get(
                "projectId"
            )

            if project_id:
                candidates.append(
                    f"{project_id}.{dataset_id}"
                )

    if len(candidates) != 1:
        raise RuntimeError(
            "Expected exactly one Controlled Tier CDR; "
            f"found {candidates}"
        )

    return candidates[0]


CDR_DATASET = resolve_aou_cdr()

bq_client = bigquery.Client(
    default_query_job_config=bigquery.QueryJobConfig(
        default_dataset=CDR_DATASET
    )
)

print("All of Us CDR:", CDR_DATASET)


## I/O and date helpers


In [ ]:
BAD_DATE_TOKENS = {
    "",
    "null",
    "none",
    "nan",
    "na",
    "n/a",
    "nat",
    "0000-00-00",
    "0000-00-00 00:00:00",
}


def parse_datetime(series):
    s = pd.Series(series)

    if pd.api.types.is_datetime64_any_dtype(
        s
    ):
        return pd.to_datetime(
            s,
            errors="coerce",
            utc=True,
        )

    s = s.astype("string").str.strip()
    s = s.where(
        ~s.str.lower().isin(
            BAD_DATE_TOKENS
        ),
        pd.NA,
    )

    return pd.to_datetime(
        s,
        errors="coerce",
        utc=True,
    )


def read_gcs_csv(path):
    with fs.open(path, "r") as f:
        return pd.read_csv(f)


def read_gcs_parquet(path):
    with fs.open(path, "rb") as f:
        return pd.read_parquet(f)


def read_gcs_pickle(path):
    with fs.open(path, "rb") as f:
        return pd.read_pickle(f)


def save_gcs_csv(df, path):
    with fs.open(path, "w") as f:
        df.to_csv(f, index=False)
    print(f"Saved CSV: {path}")


def save_gcs_pickle(obj, path):
    import pickle

    with fs.open(path, "wb") as f:
        pickle.dump(obj, f)

    print(f"Saved pickle: {path}")


def save_gcs_json(obj, path):
    with fs.open(path, "w") as f:
        json.dump(obj, f, indent=2)

    print(f"Saved JSON: {path}")


# 2. Load Frozen One-Year Development Inputs


In [ ]:
DATA_PATH = (
    f"{COHORT_DIR}/"
    f"DKD_Risk_FinalModel_Data_Landmark_1y_{FINAL_DATE}.pkl"
)

FEATURE_MANIFEST_PATH = (
    f"{RESULT_DIR}/"
    f"final_model_feature_manifest_{FINAL_DATE}.csv"
)

SPLIT_MANIFEST_PATH = (
    f"{RESULT_DIR}/"
    f"final_model_split_manifest_{FINAL_DATE}.csv"
)

BEST_PARAMS_PATH = (
    f"{RESULT_DIR}/"
    f"xgb_best_hyperparameters_{FINAL_DATE}.csv"
)

SOURCE_COHORT_PATH = (
    f"{COHORT_DIR}/"
    "aou_t2d_source_cohort.parquet"
)

PRIMARY_THRESHOLD_PATH = (
    f"{RESULT_DIR}/"
    f"fixed_policy_thresholds_cal_{FINAL_DATE}.csv"
)

paths = [
    DATA_PATH,
    FEATURE_MANIFEST_PATH,
    SPLIT_MANIFEST_PATH,
    BEST_PARAMS_PATH,
    SOURCE_COHORT_PATH,
    PRIMARY_THRESHOLD_PATH,
]

for path in paths:
    if not fs.exists(path):
        raise FileNotFoundError(path)

df_1y = read_gcs_pickle(
    DATA_PATH
)

feature_manifest = read_gcs_csv(
    FEATURE_MANIFEST_PATH
)

split_manifest = read_gcs_csv(
    SPLIT_MANIFEST_PATH
)

best_params_df = read_gcs_csv(
    BEST_PARAMS_PATH
)

source_cohort = read_gcs_parquet(
    SOURCE_COHORT_PATH
)

primary_thresholds = read_gcs_csv(
    PRIMARY_THRESHOLD_PATH
)

df_1y[ID_COL] = pd.to_numeric(
    df_1y[ID_COL],
    errors="raise",
).astype("int64")

source_cohort[ID_COL] = pd.to_numeric(
    source_cohort[ID_COL],
    errors="raise",
).astype("int64")

split_1y = (
    split_manifest.loc[
        split_manifest["landmark"].eq("1y")
    ]
    .copy()
)

split_1y[ID_COL] = pd.to_numeric(
    split_1y[ID_COL],
    errors="raise",
).astype("int64")

if set(df_1y[ID_COL]) != set(
    split_1y[ID_COL]
):
    raise ValueError(
        "One-year modeling data and split manifest do not match."
    )

print(
    f"Primary 1y cohort: N={len(df_1y):,} | "
    f"cases={int(df_1y[PRIMARY_Y].sum()):,} | "
    f"controls={int((df_1y[PRIMARY_Y] == 0).sum()):,}"
)


## Load model-specific features and primary hyperparameters


In [ ]:
model_features = {}

for model_id in MODEL_ORDER:
    features = (
        feature_manifest.loc[
            feature_manifest[
                "landmark"
            ].eq("1y")
            & feature_manifest[
                "model_id"
            ].eq(model_id)
        ]
        .sort_values(
            "feature_order"
        )[
            "feature"
        ]
        .tolist()
    )

    if not features:
        raise ValueError(
            f"No feature manifest for 1y/{model_id}."
        )

    missing = [
        feature
        for feature in features
        if feature not in df_1y.columns
    ]

    if missing:
        raise ValueError(
            f"{model_id}: missing features: {missing}"
        )

    model_features[
        model_id
    ] = features

    print(
        f"{model_id}: {len(features):,} predictors"
    )


def get_primary_xgb_params(model_id):
    row = best_params_df.loc[
        best_params_df["landmark"].eq("1y")
        & best_params_df["model_id"].eq(model_id)
    ]

    if len(row) != 1:
        raise ValueError(
            f"Expected one best-parameter row for 1y/{model_id}."
        )

    row = row.iloc[0]

    return {
        "gamma": float(row["gamma"]),
        "max_depth": int(row["max_depth"]),
        "min_child_weight": int(
            row["min_child_weight"]
        ),
        "subsample": float(row["subsample"]),
        "colsample_bytree": float(
            row["colsample_bytree"]
        ),
        "learning_rate": float(
            row["learning_rate"]
        ),
        "n_estimators": int(
            row["n_estimators"]
        ),
        "reg_alpha": float(row["reg_alpha"]),
        "reg_lambda": float(
            row["reg_lambda"]
        ),
        "scale_pos_weight": float(
            row["scale_pos_weight"]
        ),
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "n_jobs": XGB_FINAL_N_JOBS,
        "random_state": SPLIT_SEED,
    }


primary_params = {
    model_id: get_primary_xgb_params(
        model_id
    )
    for model_id in MODEL_ORDER
}


# 3. Attach Cohort Timing and Kidney-Event Metadata


In [ ]:
metadata_cols = [
    ID_COL,
    "t2d_first_evidence_date",
    "ehr_start_date",
    "ehr_end_date",
    "egfr_onset_date",
    "uacr_onset_date",
    "dkd_icd_date",
    "dialysis_code_date",
    "dialysis_lab_date",
    "transplant_date",
    "kidney_first_evidence_date",
    "kidney_first_evidence_source",
]

available_metadata_cols = [
    c for c in metadata_cols
    if c in source_cohort.columns
]

meta = (
    source_cohort[
        available_metadata_cols
    ]
    .drop_duplicates(ID_COL)
    .copy()
)

for col in [
    "t2d_first_evidence_date",
    "ehr_start_date",
    "ehr_end_date",
    "egfr_onset_date",
    "uacr_onset_date",
    "dkd_icd_date",
    "dialysis_code_date",
    "dialysis_lab_date",
    "transplant_date",
    "kidney_first_evidence_date",
]:
    if col in meta.columns:
        meta[col] = parse_datetime(
            meta[col]
        )

df = df_1y.merge(
    meta,
    on=ID_COL,
    how="left",
    validate="one_to_one",
)

df = df.merge(
    split_1y[
        [
            ID_COL,
            "split",
        ]
    ],
    on=ID_COL,
    how="left",
    validate="one_to_one",
)

df["cutoff_date"] = (
    df["t2d_first_evidence_date"]
    + pd.Timedelta(
        days=LANDMARK_DAYS
    )
)

df["followup_days_from_t2d"] = (
    df["ehr_end_date"]
    - df["t2d_first_evidence_date"]
).dt.days

df["followup_days_after_landmark"] = (
    df["ehr_end_date"]
    - df["cutoff_date"]
).dt.days

if df["split"].isna().any():
    raise ValueError(
        "At least one one-year participant is missing a split."
    )

print(
    df[
        [
            "followup_days_from_t2d",
            "followup_days_after_landmark",
        ]
    ].describe()
)


# 4. Load Longitudinal Kidney Measurements


In [ ]:
EGFR_LONG_PATH = (
    f"{DATA_CURATED_DIR}/"
    "longitudinal_egfr_measurements.parquet"
)

UACR_LONG_PATH = (
    f"{DATA_CURATED_DIR}/"
    "longitudinal_uacr_measurements.parquet"
)

egfr_long = read_gcs_parquet(
    EGFR_LONG_PATH
)

uacr_long = read_gcs_parquet(
    UACR_LONG_PATH
)

egfr_long[ID_COL] = pd.to_numeric(
    egfr_long[ID_COL],
    errors="coerce",
).astype("Int64")

uacr_long[ID_COL] = pd.to_numeric(
    uacr_long[ID_COL],
    errors="coerce",
).astype("Int64")

egfr_long["measurement_datetime"] = parse_datetime(
    egfr_long["measurement_datetime"]
)

uacr_long["measurement_datetime"] = parse_datetime(
    uacr_long["measurement_datetime"]
)

egfr_long["egfr_ml_min_1p73m2"] = pd.to_numeric(
    egfr_long["egfr_ml_min_1p73m2"],
    errors="coerce",
)

uacr_long["uacr_mg_g"] = pd.to_numeric(
    uacr_long["uacr_mg_g"],
    errors="coerce",
)

primary_ids = set(
    df[ID_COL].astype(int)
)

egfr_long = egfr_long.loc[
    egfr_long[ID_COL].astype("Int64").isin(
        primary_ids
    )
].copy()

uacr_long = uacr_long.loc[
    uacr_long[ID_COL].astype("Int64").isin(
        primary_ids
    )
].copy()

print(
    f"eGFR longitudinal rows for primary cohort: "
    f"{len(egfr_long):,}"
)

print(
    f"UACR longitudinal rows for primary cohort: "
    f"{len(uacr_long):,}"
)


# 5. Shared Evaluation and Sensitivity-Refit Helpers


In [ ]:
def choose_threshold_for_specificity(
    y,
    p,
    target_specificity=0.90,
):
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)

    fpr, tpr, thresholds = roc_curve(
        y,
        p,
    )

    specificity = 1.0 - fpr

    eligible = np.where(
        specificity
        >= target_specificity
    )[0]

    if len(eligible) == 0:
        raise RuntimeError(
            "No threshold achieves target specificity."
        )

    max_tpr = np.max(
        tpr[eligible]
    )

    candidates = eligible[
        np.isclose(
            tpr[eligible],
            max_tpr,
            rtol=0,
            atol=1e-12,
        )
    ]

    idx = candidates[
        np.argmin(
            thresholds[candidates]
        )
    ]

    return float(
        thresholds[idx]
    )


def confusion_metrics(
    y,
    p,
    threshold,
):
    y = np.asarray(y).astype(int)
    pred = (
        np.asarray(p)
        >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y,
        pred,
        labels=[0, 1],
    ).ravel()

    return {
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
        "Sensitivity": (
            tp / (tp + fn)
            if tp + fn
            else np.nan
        ),
        "Specificity": (
            tn / (tn + fp)
            if tn + fp
            else np.nan
        ),
        "PPV": (
            tp / (tp + fp)
            if tp + fp
            else np.nan
        ),
        "NPV": (
            tn / (tn + fn)
            if tn + fn
            else np.nan
        ),
    }


def stratified_bootstrap_indices(
    y,
    rng,
):
    y = np.asarray(y).astype(int)

    idx0 = np.where(
        y == 0
    )[0]

    idx1 = np.where(
        y == 1
    )[0]

    b0 = rng.choice(
        idx0,
        size=len(idx0),
        replace=True,
    )

    b1 = rng.choice(
        idx1,
        size=len(idx1),
        replace=True,
    )

    out = np.concatenate(
        [b0, b1]
    )

    rng.shuffle(out)

    return out


def percentile_ci(
    values,
    alpha=0.05,
):
    x = np.asarray(
        values,
        dtype=float,
    )

    x = x[
        np.isfinite(x)
    ]

    if len(x) == 0:
        return np.nan, np.nan

    return (
        float(
            np.quantile(
                x,
                alpha / 2,
            )
        ),
        float(
            np.quantile(
                x,
                1 - alpha / 2,
            )
        ),
    )


def bootstrap_core_metrics(
    y,
    p,
    threshold,
    *,
    n_boot=N_BOOT,
    seed=SPLIT_SEED,
):
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)

    rng = np.random.default_rng(
        seed
    )

    out = {
        "AUROC": [],
        "AUPRC": [],
        "Sensitivity": [],
        "Specificity": [],
    }

    for _ in range(n_boot):
        idx = stratified_bootstrap_indices(
            y,
            rng,
        )

        yb = y[idx]
        pb = p[idx]

        out["AUROC"].append(
            roc_auc_score(
                yb,
                pb,
            )
        )

        out["AUPRC"].append(
            average_precision_score(
                yb,
                pb,
            )
        )

        cm = confusion_metrics(
            yb,
            pb,
            threshold,
        )

        out["Sensitivity"].append(
            cm["Sensitivity"]
        )

        out["Specificity"].append(
            cm["Specificity"]
        )

    return {
        metric: percentile_ci(
            values
        )
        for metric, values in out.items()
    }


def prepare_scenario(
    base_df,
    *,
    scenario_name,
    label,
    eligible,
):
    x = base_df.loc[
        np.asarray(
            eligible,
            dtype=bool,
        )
    ].copy()

    label = pd.Series(
        label,
        index=base_df.index,
    )

    x["scenario_y"] = (
        label.loc[
            x.index
        ]
        .astype(int)
    )

    x["scenario_name"] = (
        scenario_name
    )

    return x


def fit_sensitivity_scenario(
    scenario_df,
    *,
    scenario_name,
    feature_map=None,
    save_predictions=True,
):
    if feature_map is None:
        feature_map = model_features

    rows = []

    for model_id in MODEL_ORDER:
        features = feature_map[
            model_id
        ]

        train = scenario_df.loc[
            scenario_df["split"].eq("train")
        ].copy()

        cal = scenario_df.loc[
            scenario_df["split"].eq("cal")
        ].copy()

        test = scenario_df.loc[
            scenario_df["split"].eq("test")
        ].copy()

        for split_name, part in [
            ("train", train),
            ("cal", cal),
            ("test", test),
        ]:
            if part["scenario_y"].nunique() != 2:
                raise ValueError(
                    f"{scenario_name}/{model_id}: "
                    f"{split_name} does not contain both classes."
                )

        X_train = train[
            features
        ].astype(
            np.float32
        )

        X_cal = cal[
            features
        ].astype(
            np.float32
        )

        X_test = test[
            features
        ].astype(
            np.float32
        )

        y_train = train[
            "scenario_y"
        ].astype(int).to_numpy()

        y_cal = cal[
            "scenario_y"
        ].astype(int).to_numpy()

        y_test = test[
            "scenario_y"
        ].astype(int).to_numpy()

        model = xgb.XGBClassifier(
            **primary_params[
                model_id
            ]
        )

        model.fit(
            X_train,
            y_train,
        )

        calibrator = CalibratedClassifierCV(
            model,
            method="isotonic",
            cv="prefit",
        )

        calibrator.fit(
            X_cal,
            y_cal,
        )

        p_cal = calibrator.predict_proba(
            X_cal
        )[:, 1]

        p_test = calibrator.predict_proba(
            X_test
        )[:, 1]

        threshold = choose_threshold_for_specificity(
            y_cal,
            p_cal,
            TARGET_SPECIFICITY,
        )

        cm = confusion_metrics(
            y_test,
            p_test,
            threshold,
        )

        cis = bootstrap_core_metrics(
            y_test,
            p_test,
            threshold,
        )

        row = {
            "scenario": scenario_name,
            "model_id": model_id,
            "n_train": len(train),
            "train_cases": int(
                y_train.sum()
            ),
            "n_cal": len(cal),
            "cal_cases": int(
                y_cal.sum()
            ),
            "n_test": len(test),
            "test_cases": int(
                y_test.sum()
            ),
            "AUROC": roc_auc_score(
                y_test,
                p_test,
            ),
            "AUPRC": average_precision_score(
                y_test,
                p_test,
            ),
            "Brier": brier_score_loss(
                y_test,
                p_test,
            ),
            "threshold_cal_90spec": threshold,
            **cm,
            "AUROC_ci_low": cis[
                "AUROC"
            ][0],
            "AUROC_ci_high": cis[
                "AUROC"
            ][1],
            "AUPRC_ci_low": cis[
                "AUPRC"
            ][0],
            "AUPRC_ci_high": cis[
                "AUPRC"
            ][1],
            "Sensitivity_ci_low": cis[
                "Sensitivity"
            ][0],
            "Sensitivity_ci_high": cis[
                "Sensitivity"
            ][1],
            "Specificity_ci_low": cis[
                "Specificity"
            ][0],
            "Specificity_ci_high": cis[
                "Specificity"
            ][1],
        }

        rows.append(row)

        if save_predictions:
            pred = pd.concat(
                [
                    pd.DataFrame(
                        {
                            ID_COL: cal[
                                ID_COL
                            ].astype(int),
                            "split": "cal",
                            "y_true": y_cal,
                            "p_calibrated": p_cal,
                        }
                    ),
                    pd.DataFrame(
                        {
                            ID_COL: test[
                                ID_COL
                            ].astype(int),
                            "split": "test",
                            "y_true": y_test,
                            "p_calibrated": p_test,
                        }
                    ),
                ],
                ignore_index=True,
            )

            save_gcs_csv(
                pred,
                f"{SENS_DIR}/"
                f"{scenario_name}_{model_id}_predictions.csv",
            )

        save_gcs_pickle(
            model,
            f"{SENS_MODEL_DIR}/"
            f"{scenario_name}_{model_id}_xgb.pkl",
        )

        save_gcs_pickle(
            calibrator,
            f"{SENS_MODEL_DIR}/"
            f"{scenario_name}_{model_id}_isotonic.pkl",
        )

        print(
            f"{scenario_name:<38} | {model_id} | "
            f"TEST N={len(test):,} | "
            f"cases={int(y_test.sum()):,} | "
            f"AUROC={row['AUROC']:.3f} | "
            f"Sens={row['Sensitivity']:.3f}"
        )

    return pd.DataFrame(
        rows
    )


# 6. Renal Predictor-Outcome Circularity Audit


In [ ]:
timing = df[
    [
        ID_COL,
        "t2d_first_evidence_date",
        "cutoff_date",
        PRIMARY_Y,
    ]
].copy()

egfr_pre = egfr_long.merge(
    timing,
    on=ID_COL,
    how="inner",
)

egfr_pre = egfr_pre.loc[
    (
        egfr_pre["measurement_datetime"]
        >= egfr_pre["t2d_first_evidence_date"]
    )
    & (
        egfr_pre["measurement_datetime"]
        < egfr_pre["cutoff_date"]
    )
].copy()

egfr_flag = (
    egfr_pre.groupby(
        ID_COL,
        as_index=False,
    )
    .agg(
        pre_egfr_n=(
            "egfr_ml_min_1p73m2",
            "count",
        ),
        pre_egfr_min=(
            "egfr_ml_min_1p73m2",
            "min",
        ),
    )
)

egfr_flag[
    "pre_egfr_abnormal"
] = (
    egfr_flag[
        "pre_egfr_min"
    ] < 60
)

uacr_pre = uacr_long.merge(
    timing,
    on=ID_COL,
    how="inner",
)

uacr_pre = uacr_pre.loc[
    (
        uacr_pre["measurement_datetime"]
        >= uacr_pre["t2d_first_evidence_date"]
    )
    & (
        uacr_pre["measurement_datetime"]
        < uacr_pre["cutoff_date"]
    )
].copy()

uacr_flag = (
    uacr_pre.groupby(
        ID_COL,
        as_index=False,
    )
    .agg(
        pre_uacr_n=(
            "uacr_mg_g",
            "count",
        ),
        pre_uacr_max=(
            "uacr_mg_g",
            "max",
        ),
    )
)

uacr_flag[
    "pre_uacr_abnormal"
] = (
    uacr_flag[
        "pre_uacr_max"
    ] >= 30
)

renal_audit = (
    df[
        [
            ID_COL,
            PRIMARY_Y,
            "split",
        ]
    ]
    .merge(
        egfr_flag,
        on=ID_COL,
        how="left",
    )
    .merge(
        uacr_flag,
        on=ID_COL,
        how="left",
    )
)

for col in [
    "pre_egfr_n",
    "pre_uacr_n",
]:
    renal_audit[col] = (
        renal_audit[col]
        .fillna(0)
        .astype(int)
    )

for col in [
    "pre_egfr_abnormal",
    "pre_uacr_abnormal",
]:
    renal_audit[col] = (
        renal_audit[col]
        .fillna(False)
        .astype(bool)
    )

renal_audit[
    "has_prelandmark_renal_measurement"
] = (
    renal_audit[
        "pre_egfr_n"
    ].gt(0)
    | renal_audit[
        "pre_uacr_n"
    ].gt(0)
)

renal_audit[
    "any_prelandmark_renal_abnormality"
] = (
    renal_audit[
        "pre_egfr_abnormal"
    ]
    | renal_audit[
        "pre_uacr_abnormal"
    ]
)

renal_audit[
    "documented_normal_prelandmark_renal"
] = (
    renal_audit[
        "has_prelandmark_renal_measurement"
    ]
    & ~renal_audit[
        "any_prelandmark_renal_abnormality"
    ]
)

renal_audit_summary = (
    renal_audit.groupby(
        PRIMARY_Y
    )
    .agg(
        n=(
            ID_COL,
            "size",
        ),
        has_prelandmark_renal_measurement=(
            "has_prelandmark_renal_measurement",
            "mean",
        ),
        any_prelandmark_renal_abnormality=(
            "any_prelandmark_renal_abnormality",
            "mean",
        ),
        documented_normal_prelandmark_renal=(
            "documented_normal_prelandmark_renal",
            "mean",
        ),
    )
    .reset_index()
)

save_gcs_csv(
    renal_audit,
    f"{SENS_DIR}/"
    "prelandmark_renal_abnormality_person_level.csv",
)

save_gcs_csv(
    renal_audit_summary,
    f"{SENS_DIR}/"
    "prelandmark_renal_abnormality_summary.csv",
)

display(
    renal_audit_summary
)


## Refit on participants with documented normal pre-landmark renal measurements


In [ ]:
renal_flags = renal_audit.set_index(
    ID_COL
)

eligible_documented_normal = (
    df[ID_COL]
    .map(
        renal_flags[
            "documented_normal_prelandmark_renal"
        ]
    )
    .fillna(False)
    .to_numpy()
)

scenario_documented_normal = prepare_scenario(
    df,
    scenario_name="documented_normal_prelandmark_renal",
    label=df[PRIMARY_Y],
    eligible=eligible_documented_normal,
)

documented_normal_results = fit_sensitivity_scenario(
    scenario_documented_normal,
    scenario_name="documented_normal_prelandmark_renal",
)

save_gcs_csv(
    documented_normal_results,
    f"{SENS_DIR}/"
    "documented_normal_prelandmark_renal_results.csv",
)


## Secondary leakage sensitivity: no observed pre-landmark renal abnormality


In [ ]:
eligible_no_abnormality = (
    ~df[ID_COL]
    .map(
        renal_flags[
            "any_prelandmark_renal_abnormality"
        ]
    )
    .fillna(False)
    .to_numpy()
)

scenario_no_abnormality = prepare_scenario(
    df,
    scenario_name="no_prelandmark_renal_abnormality",
    label=df[PRIMARY_Y],
    eligible=eligible_no_abnormality,
)

no_abnormality_results = fit_sensitivity_scenario(
    scenario_no_abnormality,
    scenario_name="no_prelandmark_renal_abnormality",
)

save_gcs_csv(
    no_abnormality_results,
    f"{SENS_DIR}/"
    "no_prelandmark_renal_abnormality_results.csv",
)


# 7. Alternative DKD Outcome Definitions


These analyses preserve the primary one-year risk set and split assignments while changing
the outcome label. They are intended as sensitivity analyses of endpoint composition rather
than replacements for the primary composite endpoint.

- Persistent eGFR-defined DKD.
- Persistent UACR-defined DKD.
- Both persistent eGFR and persistent UACR.
- DKD-specific diagnosis-code-only ascertainment.


In [ ]:
def after_landmark(date_col):
    return (
        df[date_col].notna()
        & (
            df[date_col]
            > df["cutoff_date"]
        )
    )


component_labels = {
    "egfr_defined_dkd": (
        after_landmark(
            "egfr_onset_date"
        )
    ),
    "uacr_defined_dkd": (
        after_landmark(
            "uacr_onset_date"
        )
    ),
    "both_egfr_and_uacr_dkd": (
        after_landmark(
            "egfr_onset_date"
        )
        & after_landmark(
            "uacr_onset_date"
        )
    ),
    "dkd_icd_only": (
        after_landmark(
            "dkd_icd_date"
        )
    ),
}

component_case_counts = pd.DataFrame(
    [
        {
            "outcome": name,
            "n": len(df),
            "cases": int(
                label.sum()
            ),
            "controls": int(
                len(df)
                - label.sum()
            ),
            "case_rate": float(
                label.mean()
            ),
        }
        for name, label in component_labels.items()
    ]
)

save_gcs_csv(
    component_case_counts,
    f"{SENS_DIR}/"
    "alternative_outcome_case_counts.csv",
)

display(
    component_case_counts
)


In [ ]:
alternative_outcome_results = []

for scenario_name, label in component_labels.items():
    scenario = prepare_scenario(
        df,
        scenario_name=scenario_name,
        label=label.astype(int),
        eligible=np.ones(
            len(df),
            dtype=bool,
        ),
    )

    result = fit_sensitivity_scenario(
        scenario,
        scenario_name=scenario_name,
    )

    alternative_outcome_results.append(
        result
    )

alternative_outcome_results_df = pd.concat(
    alternative_outcome_results,
    ignore_index=True,
)

save_gcs_csv(
    alternative_outcome_results_df,
    f"{SENS_DIR}/"
    "alternative_outcome_model_results.csv",
)


# 8. Longer Minimum Follow-up for Primary Controls


This sensitivity preserves all primary cases and progressively requires primary controls to
have at least 3 or 4 years of EHR observation after the first documented T2D date.


In [ ]:
control_followup_results = []

for years, days in [
    (3, 365 * 3),
    (4, 365 * 4),
]:
    eligible = (
        df[PRIMARY_Y].eq(1)
        | (
            df[PRIMARY_Y].eq(0)
            & df[
                "followup_days_from_t2d"
            ].ge(days)
        )
    )

    scenario_name = (
        f"control_min_followup_{years}y_from_t2d"
    )

    scenario = prepare_scenario(
        df,
        scenario_name=scenario_name,
        label=df[PRIMARY_Y],
        eligible=eligible,
    )

    result = fit_sensitivity_scenario(
        scenario,
        scenario_name=scenario_name,
    )

    control_followup_results.append(
        result
    )

control_followup_results_df = pd.concat(
    control_followup_results,
    ignore_index=True,
)

save_gcs_csv(
    control_followup_results_df,
    f"{SENS_DIR}/"
    "longer_control_followup_results.csv",
)


# 9. Death / Competing-Event Data


In [ ]:
death_available = True

try:
    death_query = f'''
    SELECT
        person_id,
        death_date
    FROM `{CDR_DATASET}.death`
    WHERE death_date IS NOT NULL
    '''

    death_df = (
        bq_client.query(
            death_query
        )
        .result()
        .to_dataframe()
    )

    death_df[ID_COL] = pd.to_numeric(
        death_df[ID_COL],
        errors="coerce",
    ).astype("Int64")

    death_df["death_date"] = parse_datetime(
        death_df["death_date"]
    )

    death_df = (
        death_df.dropna(
            subset=[
                ID_COL,
                "death_date",
            ]
        )
        .groupby(
            ID_COL,
            as_index=False,
        )[
            "death_date"
        ]
        .min()
    )

    death_df[ID_COL] = death_df[
        ID_COL
    ].astype("int64")

    print(
        f"Death records loaded: {len(death_df):,}"
    )

except Exception as exc:
    death_available = False

    death_df = pd.DataFrame(
        columns=[
            ID_COL,
            "death_date",
        ]
    )

    print(
        "Death table could not be loaded. "
        "EHR-observation fixed-horizon analyses will still run."
    )

    print(
        "Reason:",
        repr(exc),
    )

df = df.merge(
    death_df,
    on=ID_COL,
    how="left",
    validate="one_to_one",
)

if death_available:
    print(
        "Deaths among primary 1y participants:",
        int(
            df[
                "death_date"
            ].notna().sum()
        ),
    )


# 10. Fixed 3-Year and 5-Year Post-Landmark Prediction Horizons


The fixed-horizon binary endpoint is defined prospectively from the one-year landmark.

For a horizon ending at `landmark + H`:

- **Case:** qualifying DKD occurs after the landmark and on/before the horizon.
- **Control:** no qualifying DKD through the horizon and EHR observation extends through
  the horizon.
- In the confirmed-survival variant, participants who die before the horizon without prior
  DKD are excluded as competing events.

Participants who develop DKD only after the horizon can therefore serve as horizon-specific
controls if they remain under observation through the horizon.


In [ ]:
fixed_horizon_results = []
fixed_horizon_counts = []

for horizon_years in [
    3,
    5,
]:
    horizon_days = (
        365
        * horizon_years
    )

    horizon_end = (
        df["cutoff_date"]
        + pd.to_timedelta(
            horizon_days,
            unit="D",
        )
    )

    kidney = df[
        "kidney_first_evidence_date"
    ]

    event_within = (
        kidney.notna()
        & (
            kidney
            > df["cutoff_date"]
        )
        & (
            kidney
            <= horizon_end
        )
    )

    no_event_through_horizon = (
        kidney.isna()
        | (
            kidney
            > horizon_end
        )
    )

    observed_through_horizon = (
        df[
            "ehr_end_date"
        ]
        >= horizon_end
    )

    # EHR-observation-only fixed horizon.
    eligible_ehr = (
        event_within
        | (
            no_event_through_horizon
            & observed_through_horizon
        )
    )

    label_ehr = (
        event_within
        .astype(int)
    )

    scenario_name = (
        f"fixed_{horizon_years}y_post_landmark_ehr"
    )

    fixed_horizon_counts.append(
        {
            "scenario": scenario_name,
            "n": int(
                eligible_ehr.sum()
            ),
            "cases": int(
                (
                    event_within
                    & eligible_ehr
                ).sum()
            ),
            "controls": int(
                (
                    eligible_ehr
                    & ~event_within
                ).sum()
            ),
        }
    )

    scenario = prepare_scenario(
        df,
        scenario_name=scenario_name,
        label=label_ehr,
        eligible=eligible_ehr,
    )

    fixed_horizon_results.append(
        fit_sensitivity_scenario(
            scenario,
            scenario_name=scenario_name,
        )
    )

    # Confirmed-survival sensitivity if death data are available.
    if death_available:
        death = df[
            "death_date"
        ]

        case_before_death = (
            event_within
            & (
                death.isna()
                | (
                    kidney
                    <= death
                )
            )
        )

        survived_horizon = (
            death.isna()
            | (
                death
                > horizon_end
            )
        )

        eligible_survival = (
            case_before_death
            | (
                no_event_through_horizon
                & observed_through_horizon
                & survived_horizon
            )
        )

        label_survival = (
            case_before_death
            .astype(int)
        )

        scenario_name_surv = (
            f"fixed_{horizon_years}y_post_landmark_survival"
        )

        fixed_horizon_counts.append(
            {
                "scenario": scenario_name_surv,
                "n": int(
                    eligible_survival.sum()
                ),
                "cases": int(
                    (
                        case_before_death
                        & eligible_survival
                    ).sum()
                ),
                "controls": int(
                    (
                        eligible_survival
                        & ~case_before_death
                    ).sum()
                ),
            }
        )

        scenario_surv = prepare_scenario(
            df,
            scenario_name=scenario_name_surv,
            label=label_survival,
            eligible=eligible_survival,
        )

        fixed_horizon_results.append(
            fit_sensitivity_scenario(
                scenario_surv,
                scenario_name=scenario_name_surv,
            )
        )

fixed_horizon_counts_df = pd.DataFrame(
    fixed_horizon_counts
)

fixed_horizon_results_df = pd.concat(
    fixed_horizon_results,
    ignore_index=True,
)

save_gcs_csv(
    fixed_horizon_counts_df,
    f"{SENS_DIR}/"
    "fixed_horizon_cohort_counts.csv",
)

save_gcs_csv(
    fixed_horizon_results_df,
    f"{SENS_DIR}/"
    "fixed_horizon_model_results.csv",
)

display(
    fixed_horizon_counts_df
)


# 11. Missingness by Information Layer and Outcome


In [ ]:
incremental_domains = {
    "M1_intrinsic": (
        model_features["M1"]
    ),
    "M2_snapshot_added": [
        f
        for f in model_features["M2"]
        if f not in set(
            model_features["M1"]
        )
    ],
    "M3_medication_added": [
        f
        for f in model_features["M3"]
        if f not in set(
            model_features["M2"]
        )
    ],
    "M4_longitudinal_added": [
        f
        for f in model_features["M4"]
        if f not in set(
            model_features["M3"]
        )
    ],
    "M5_social_added": [
        f
        for f in model_features["M5"]
        if f not in set(
            model_features["M4"]
        )
    ],
}

missingness_rows = []

for domain, features in incremental_domains.items():
    x = df[
        [
            PRIMARY_Y,
        ]
        + features
    ].copy()

    for outcome_value, group in x.groupby(
        PRIMARY_Y
    ):
        vals = group[
            features
        ]

        participant_missing_fraction = (
            vals.isna().mean(
                axis=1
            )
        )

        missingness_rows.append(
            {
                "domain": domain,
                "outcome": int(
                    outcome_value
                ),
                "n": len(group),
                "n_features": len(
                    features
                ),
                "cell_missing_fraction": float(
                    vals.isna().to_numpy().mean()
                ),
                "mean_participant_missing_fraction": float(
                    participant_missing_fraction.mean()
                ),
                "median_participant_missing_fraction": float(
                    participant_missing_fraction.median()
                ),
                "participants_all_features_missing_fraction": float(
                    vals.isna().all(
                        axis=1
                    ).mean()
                ),
            }
        )

missingness_df = pd.DataFrame(
    missingness_rows
)

save_gcs_csv(
    missingness_df,
    f"{SENS_DIR}/"
    "missingness_by_incremental_layer_and_outcome.csv",
)

display(
    missingness_df
)


# 12. Pre-Landmark Visit and Measurement Density


In [ ]:
GENERAL_DOMAINS = {
    "glycemia": [
        "HbA1c",
        "RBG",
    ],
    "blood_pressure": [
        "SBP",
        "DBP",
    ],
    "lipids": [
        "LDL",
        "HDL",
        "TG",
        "TotalChol",
    ],
    "electrolytes": [
        "Na",
        "K",
        "Cl",
        "HCO3",
    ],
    "liver_protein": [
        "ALT",
        "AST",
        "ALP",
        "Albumin",
        "TotalBilirubin",
        "DirectBilirubin",
        "TotalProtein",
    ],
    "hematology": [
        "Hgb",
    ],
    "anthropometrics": [
        "Weight",
    ],
    "vitals": [
        "HR",
    ],
}


def build_observability_scores(
    feature_df,
):
    out = feature_df[
        [
            ID_COL,
        ]
    ].copy()

    used_n_cols = []
    domain_flags = {}

    for domain, bases in GENERAL_DOMAINS.items():
        n_cols = [
            f"{base}_n"
            for base in bases
            if f"{base}_n"
            in feature_df.columns
        ]

        used_n_cols.extend(
            n_cols
        )

        if not n_cols:
            domain_flags[
                domain
            ] = pd.Series(
                0,
                index=feature_df.index,
                dtype=int,
            )

        else:
            x = (
                feature_df[
                    n_cols
                ]
                .apply(
                    pd.to_numeric,
                    errors="coerce",
                )
                .fillna(0)
            )

            domain_flags[
                domain
            ] = (
                x.sum(
                    axis=1
                )
                > 0
            ).astype(int)

    flags = pd.DataFrame(
        domain_flags
    )

    out[
        "obs_general_any"
    ] = flags.sum(
        axis=1
    ).astype(int)

    used_n_cols = list(
        dict.fromkeys(
            used_n_cols
        )
    )

    if used_n_cols:
        out[
            "obs_general_count"
        ] = (
            feature_df[
                used_n_cols
            ]
            .apply(
                pd.to_numeric,
                errors="coerce",
            )
            .fillna(0)
            .sum(
                axis=1
            )
        )

    else:
        out[
            "obs_general_count"
        ] = 0.0

    return out, used_n_cols


observability, observability_count_cols = build_observability_scores(
    df
)

df = df.merge(
    observability,
    on=ID_COL,
    how="left",
    validate="one_to_one",
)

print(
    "Observability *_n features used:",
    observability_count_cols,
)


## Query visit occurrence counts before the one-year landmark


In [ ]:
person_ids = (
    df[
        ID_COL
    ]
    .astype(int)
    .tolist()
)

visit_query = f'''
SELECT
    person_id,
    visit_occurrence_id,
    visit_start_datetime
FROM `{CDR_DATASET}.visit_occurrence`
WHERE
    person_id IN UNNEST(@person_ids)
    AND visit_start_datetime IS NOT NULL
'''

visit_job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ArrayQueryParameter(
            "person_ids",
            "INT64",
            person_ids,
        )
    ]
)

visits = (
    bq_client.query(
        visit_query,
        job_config=visit_job_config,
    )
    .result()
    .to_dataframe()
)

visits[ID_COL] = pd.to_numeric(
    visits[ID_COL],
    errors="coerce",
).astype("Int64")

visits[
    "visit_start_datetime"
] = parse_datetime(
    visits[
        "visit_start_datetime"
    ]
)

visit_timing = df[
    [
        ID_COL,
        "t2d_first_evidence_date",
        "cutoff_date",
    ]
]

visits = visits.merge(
    visit_timing,
    on=ID_COL,
    how="inner",
)

visits_pre = visits.loc[
    (
        visits[
            "visit_start_datetime"
        ]
        >= visits[
            "t2d_first_evidence_date"
        ]
    )
    & (
        visits[
            "visit_start_datetime"
        ]
        < visits[
            "cutoff_date"
        ]
    )
].copy()

visit_counts = (
    visits_pre.groupby(
        ID_COL,
        as_index=False,
    )[
        "visit_occurrence_id"
    ]
    .nunique()
    .rename(
        columns={
            "visit_occurrence_id":
            "prelandmark_visit_count"
        }
    )
)

df = df.merge(
    visit_counts,
    on=ID_COL,
    how="left",
    validate="one_to_one",
)

df[
    "prelandmark_visit_count"
] = (
    df[
        "prelandmark_visit_count"
    ]
    .fillna(0)
    .astype(int)
)

visit_measurement_summary = (
    df.groupby(
        PRIMARY_Y
    )[
        [
            "prelandmark_visit_count",
            "obs_general_any",
            "obs_general_count",
        ]
    ]
    .agg(
        [
            "count",
            "mean",
            "median",
            "std",
        ]
    )
)

visit_measurement_summary.columns = [
    "_".join(
        col
    )
    for col in visit_measurement_summary.columns
]

visit_measurement_summary = (
    visit_measurement_summary
    .reset_index()
)

save_gcs_csv(
    visit_measurement_summary,
    f"{SENS_DIR}/"
    "visit_measurement_density_by_outcome.csv",
)

display(
    visit_measurement_summary
)


# 13. Post-Landmark Kidney Monitoring / Ascertainment Intensity


In [ ]:
monitor_timing = df[
    [
        ID_COL,
        PRIMARY_Y,
        "cutoff_date",
        "kidney_first_evidence_date",
        "ehr_end_date",
    ]
].copy()

monitor_timing[
    "monitor_stop_date"
] = np.where(
    monitor_timing[
        PRIMARY_Y
    ].eq(1),
    monitor_timing[
        "kidney_first_evidence_date"
    ],
    monitor_timing[
        "ehr_end_date"
    ],
)

monitor_timing[
    "monitor_stop_date"
] = parse_datetime(
    monitor_timing[
        "monitor_stop_date"
    ]
)

post_egfr = egfr_long.merge(
    monitor_timing[
        [
            ID_COL,
            "cutoff_date",
            "monitor_stop_date",
        ]
    ],
    on=ID_COL,
    how="inner",
)

post_egfr = post_egfr.loc[
    (
        post_egfr[
            "measurement_datetime"
        ]
        >= post_egfr[
            "cutoff_date"
        ]
    )
    & (
        post_egfr[
            "measurement_datetime"
        ]
        <= post_egfr[
            "monitor_stop_date"
        ]
    )
]

post_uacr = uacr_long.merge(
    monitor_timing[
        [
            ID_COL,
            "cutoff_date",
            "monitor_stop_date",
        ]
    ],
    on=ID_COL,
    how="inner",
)

post_uacr = post_uacr.loc[
    (
        post_uacr[
            "measurement_datetime"
        ]
        >= post_uacr[
            "cutoff_date"
        ]
    )
    & (
        post_uacr[
            "measurement_datetime"
        ]
        <= post_uacr[
            "monitor_stop_date"
        ]
    )
]

egfr_counts = (
    post_egfr.groupby(
        ID_COL
    )
    .size()
    .rename(
        "postlandmark_egfr_count"
    )
)

uacr_counts = (
    post_uacr.groupby(
        ID_COL
    )
    .size()
    .rename(
        "postlandmark_uacr_count"
    )
)

monitor = monitor_timing.merge(
    egfr_counts,
    on=ID_COL,
    how="left",
).merge(
    uacr_counts,
    on=ID_COL,
    how="left",
)

for col in [
    "postlandmark_egfr_count",
    "postlandmark_uacr_count",
]:
    monitor[col] = (
        monitor[col]
        .fillna(0)
        .astype(int)
    )

monitor[
    "monitor_years"
] = (
    (
        monitor[
            "monitor_stop_date"
        ]
        - monitor[
            "cutoff_date"
        ]
    ).dt.days
    / 365.25
)

monitor[
    "monitor_years"
] = monitor[
    "monitor_years"
].clip(
    lower=1 / 365.25
)

monitor[
    "postlandmark_kidney_measurement_count"
] = (
    monitor[
        "postlandmark_egfr_count"
    ]
    + monitor[
        "postlandmark_uacr_count"
    ]
)

monitor[
    "postlandmark_kidney_measurements_per_year"
] = (
    monitor[
        "postlandmark_kidney_measurement_count"
    ]
    / monitor[
        "monitor_years"
    ]
)

monitor_summary = (
    monitor.groupby(
        PRIMARY_Y
    )[
        [
            "postlandmark_egfr_count",
            "postlandmark_uacr_count",
            "postlandmark_kidney_measurement_count",
            "postlandmark_kidney_measurements_per_year",
        ]
    ]
    .agg(
        [
            "mean",
            "median",
            "std",
        ]
    )
)

monitor_summary.columns = [
    "_".join(
        col
    )
    for col in monitor_summary.columns
]

monitor_summary = (
    monitor_summary
    .reset_index()
)

save_gcs_csv(
    monitor,
    f"{SENS_DIR}/"
    "postlandmark_kidney_monitoring_person_level.csv",
)

save_gcs_csv(
    monitor_summary,
    f"{SENS_DIR}/"
    "postlandmark_kidney_monitoring_by_outcome.csv",
)

display(
    monitor_summary
)


# 14. Primary Model Performance Across Pre-Landmark Observation Density


In [ ]:
def make_train_based_tertiles(
    values,
    split,
):
    values = pd.to_numeric(
        values,
        errors="coerce",
    ).fillna(0)

    train_values = values[
        split.eq("train")
    ]

    q1, q2 = np.quantile(
        train_values,
        [
            1 / 3,
            2 / 3,
        ],
    )

    # Guarantee strictly increasing cut points.
    if q2 <= q1:
        ranked = train_values.rank(
            method="average",
            pct=True,
        )

        q1 = float(
            train_values.loc[
                ranked.sub(
                    1 / 3
                ).abs().idxmin()
            ]
        )

        q2 = float(
            train_values.loc[
                ranked.sub(
                    2 / 3
                ).abs().idxmin()
            ]
        )

    groups = pd.cut(
        values,
        bins=[
            -np.inf,
            q1,
            q2,
            np.inf,
        ],
        labels=[
            "Low",
            "Intermediate",
            "High",
        ],
        include_lowest=True,
    )

    return groups, q1, q2


df[
    "observation_density_tertile"
], density_q1, density_q2 = make_train_based_tertiles(
    df[
        "obs_general_count"
    ],
    df[
        "split"
    ],
)

print(
    "Training-derived observation-density cut points:",
    density_q1,
    density_q2,
)

prediction_sets = {}

for model_id in MODEL_ORDER:
    path = (
        f"{RESULT_DIR}/"
        f"dkd_1y_{model_id}_predictions_{FINAL_DATE}.csv"
    )

    prediction_sets[
        model_id
    ] = read_gcs_csv(
        path
    )

density_test = df.loc[
    df[
        "split"
    ].eq("test"),
    [
        ID_COL,
        PRIMARY_Y,
        "observation_density_tertile",
        "obs_general_any",
        "obs_general_count",
        "prelandmark_visit_count",
    ],
].copy()

density_performance_rows = []

for model_id in MODEL_ORDER:
    pred = (
        prediction_sets[
            model_id
        ]
        .loc[
            prediction_sets[
                model_id
            ][
                "split"
            ].eq("test")
        ]
        .copy()
    )

    pred[ID_COL] = pd.to_numeric(
        pred[ID_COL],
        errors="raise",
    ).astype("int64")

    x = density_test.merge(
        pred[
            [
                ID_COL,
                "y_true",
                "p_calibrated",
            ]
        ],
        on=ID_COL,
        how="inner",
        validate="one_to_one",
    )

    threshold = float(
        primary_thresholds.loc[
            primary_thresholds[
                "landmark"
            ].eq("1y")
            & primary_thresholds[
                "model_id"
            ].eq(model_id),
            "threshold",
        ].iloc[0]
    )

    for density_level, g in x.groupby(
        "observation_density_tertile",
        observed=True,
    ):
        y = g[
            "y_true"
        ].astype(int).to_numpy()

        p = g[
            "p_calibrated"
        ].to_numpy()

        cm = confusion_metrics(
            y,
            p,
            threshold,
        )

        density_performance_rows.append(
            {
                "model_id": model_id,
                "observation_density": str(
                    density_level
                ),
                "n": len(g),
                "cases": int(
                    y.sum()
                ),
                "case_rate": float(
                    y.mean()
                ),
                "AUROC": (
                    roc_auc_score(
                        y,
                        p,
                    )
                    if np.unique(
                        y
                    ).size == 2
                    else np.nan
                ),
                "AUPRC": (
                    average_precision_score(
                        y,
                        p,
                    )
                    if np.unique(
                        y
                    ).size == 2
                    else np.nan
                ),
                **cm,
            }
        )

density_performance_df = pd.DataFrame(
    density_performance_rows
)

save_gcs_csv(
    density_performance_df,
    f"{SENS_DIR}/"
    "performance_by_prelandmark_observation_density.csv",
)

display(
    density_performance_df
)


# 15. PRS-Free Nested Models


In [ ]:
def is_prs_feature(feature):
    s = str(
        feature
    ).upper()

    return (
        "_PRS_" in s
        or s.startswith(
            "PRS_"
        )
        or s.endswith(
            "_PRS"
        )
        or "PGS" in s
    )


prs_free_feature_map = {}

prs_feature_rows = []

for model_id in MODEL_ORDER:
    original = model_features[
        model_id
    ]

    removed = [
        f
        for f in original
        if is_prs_feature(
            f
        )
    ]

    retained = [
        f
        for f in original
        if f not in set(
            removed
        )
    ]

    prs_free_feature_map[
        model_id
    ] = retained

    for f in removed:
        prs_feature_rows.append(
            {
                "model_id": model_id,
                "removed_prs_feature": f,
            }
        )

    print(
        f"{model_id}: removed {len(removed)} PRS features; "
        f"retained {len(retained)} predictors"
    )

prs_removed_df = pd.DataFrame(
    prs_feature_rows
)

save_gcs_csv(
    prs_removed_df,
    f"{SENS_DIR}/"
    "prs_features_removed_by_model.csv",
)

prs_free_scenario = prepare_scenario(
    df,
    scenario_name="prs_free_primary_outcome",
    label=df[PRIMARY_Y],
    eligible=np.ones(
        len(df),
        dtype=bool,
    ),
)

prs_free_results = fit_sensitivity_scenario(
    prs_free_scenario,
    scenario_name="prs_free_primary_outcome",
    feature_map=prs_free_feature_map,
)

save_gcs_csv(
    prs_free_results,
    f"{SENS_DIR}/"
    "prs_free_nested_model_results.csv",
)


# 16. Included-versus-Excluded Cohort Characteristics


Two descriptive selection-bias summaries are produced:

1. Adult participants with valid EHR coverage around the first documented T2D date,
   stratified by whether they satisfy the 365-day pre-index observation requirement.
2. Participants in the corrected T2D source cohort, stratified by whether they enter the
   final one-year analytic risk set.

These tables are descriptive and are not used for model fitting.


In [ ]:
T2D_PATH = (
    f"{DATA_CURATED_DIR}/"
    "final_t2d_first_evidence.csv"
)

EHR_PATH = (
    f"{DATA_CURATED_DIR}/"
    "derived_ehr_dates.csv"
)

DEMOGRAPHICS_PATH = (
    f"{DATA_CURATED_DIR}/"
    "participant_demographics.parquet"
)

t2d_all = read_gcs_csv(
    T2D_PATH
)

ehr_all = read_gcs_csv(
    EHR_PATH
)

demographics = read_gcs_parquet(
    DEMOGRAPHICS_PATH
)

for x in [
    t2d_all,
    ehr_all,
    demographics,
]:
    x[ID_COL] = pd.to_numeric(
        x[ID_COL],
        errors="coerce",
    ).astype("Int64")

t2d_all[
    "t2d_first_evidence_date"
] = parse_datetime(
    t2d_all[
        "t2d_first_evidence_date"
    ]
)

ehr_all[
    "ehr_start_date"
] = parse_datetime(
    ehr_all[
        "ehr_start_date"
    ]
)

ehr_all[
    "ehr_end_date"
] = parse_datetime(
    ehr_all[
        "ehr_end_date"
    ]
)

demo_cols = [
    ID_COL,
    "date_of_birth",
]

for candidate in [
    "sex_at_birth",
    "gender_identity",
    "race",
    "ethnicity",
]:
    if candidate in demographics.columns:
        demo_cols.append(
            candidate
        )

demo = (
    demographics[
        list(
            dict.fromkeys(
                demo_cols
            )
        )
    ]
    .drop_duplicates(
        ID_COL
    )
    .copy()
)

demo[
    "date_of_birth"
] = parse_datetime(
    demo[
        "date_of_birth"
    ]
)

preindex = (
    t2d_all[
        [
            ID_COL,
            "t2d_first_evidence_date",
        ]
    ]
    .drop_duplicates(
        ID_COL
    )
    .merge(
        ehr_all[
            [
                ID_COL,
                "ehr_start_date",
                "ehr_end_date",
            ]
        ].drop_duplicates(
            ID_COL
        ),
        on=ID_COL,
        how="left",
    )
    .merge(
        demo,
        on=ID_COL,
        how="left",
    )
)

preindex[
    "age_at_t2d"
] = (
    (
        preindex[
            "t2d_first_evidence_date"
        ]
        - preindex[
            "date_of_birth"
        ]
    ).dt.days
    / 365.25
)

preindex[
    "washout_days"
] = (
    preindex[
        "t2d_first_evidence_date"
    ]
    - preindex[
        "ehr_start_date"
    ]
).dt.days

preindex[
    "ehr_span_days"
] = (
    preindex[
        "ehr_end_date"
    ]
    - preindex[
        "ehr_start_date"
    ]
).dt.days

preindex[
    "valid_adult_ehr"
] = (
    preindex[
        "age_at_t2d"
    ].ge(18)
    & preindex[
        "ehr_start_date"
    ].notna()
    & preindex[
        "ehr_end_date"
    ].notna()
    & (
        preindex[
            "ehr_start_date"
        ]
        <= preindex[
            "t2d_first_evidence_date"
        ]
    )
    & (
        preindex[
            "ehr_end_date"
        ]
        >= preindex[
            "t2d_first_evidence_date"
        ]
    )
)

preindex = preindex.loc[
    preindex[
        "valid_adult_ehr"
    ]
].copy()

preindex[
    "passes_365d_washout"
] = (
    preindex[
        "washout_days"
    ]
    >= 365
)


def continuous_group_summary(
    data,
    group_col,
    variables,
):
    rows = []

    for group_value, g in data.groupby(
        group_col
    ):
        for variable in variables:
            x = pd.to_numeric(
                g[
                    variable
                ],
                errors="coerce",
            ).dropna()

            rows.append(
                {
                    "group_variable": group_col,
                    "group": str(
                        group_value
                    ),
                    "variable": variable,
                    "n": len(x),
                    "mean": (
                        float(
                            x.mean()
                        )
                        if len(x)
                        else np.nan
                    ),
                    "sd": (
                        float(
                            x.std()
                        )
                        if len(x) > 1
                        else np.nan
                    ),
                    "median": (
                        float(
                            x.median()
                        )
                        if len(x)
                        else np.nan
                    ),
                    "q1": (
                        float(
                            x.quantile(
                                0.25
                            )
                        )
                        if len(x)
                        else np.nan
                    ),
                    "q3": (
                        float(
                            x.quantile(
                                0.75
                            )
                        )
                        if len(x)
                        else np.nan
                    ),
                }
            )

    return pd.DataFrame(
        rows
    )


preindex_continuous = continuous_group_summary(
    preindex,
    "passes_365d_washout",
    [
        "age_at_t2d",
        "washout_days",
        "ehr_span_days",
    ],
)

save_gcs_csv(
    preindex_continuous,
    f"{SENS_DIR}/"
    "selection_bias_preindex_washout_continuous.csv",
)

# One-year inclusion among corrected source cohort.
source_selection = source_cohort.copy()

for col in [
    "t2d_first_evidence_date",
    "ehr_start_date",
    "ehr_end_date",
]:
    source_selection[
        col
    ] = parse_datetime(
        source_selection[
            col
        ]
    )

source_selection[
    "followup_days_from_t2d"
] = (
    source_selection[
        "ehr_end_date"
    ]
    - source_selection[
        "t2d_first_evidence_date"
    ]
).dt.days

source_selection[
    "ehr_span_days"
] = (
    source_selection[
        "ehr_end_date"
    ]
    - source_selection[
        "ehr_start_date"
    ]
).dt.days

included_1y_ids = set(
    df[
        ID_COL
    ].astype(int)
)

source_selection[
    "included_1y"
] = (
    source_selection[
        ID_COL
    ]
    .astype(int)
    .isin(
        included_1y_ids
    )
)

source_continuous = continuous_group_summary(
    source_selection,
    "included_1y",
    [
        "age_at_t2d",
        "washout_days",
        "followup_days_from_t2d",
        "ehr_span_days",
    ],
)

save_gcs_csv(
    source_continuous,
    f"{SENS_DIR}/"
    "selection_bias_source_vs_1y_included_continuous.csv",
)

# Optional categorical summaries when curated demographic fields are available.
categorical_rows = []

for variable in [
    "sex_at_birth",
    "gender_identity",
    "race",
    "ethnicity",
]:
    if variable not in preindex.columns:
        continue

    x = preindex[
        [
            "passes_365d_washout",
            variable,
        ]
    ].copy()

    x[
        variable
    ] = x[
        variable
    ].astype(
        "string"
    ).fillna(
        "Missing"
    )

    counts = (
        x.groupby(
            [
                "passes_365d_washout",
                variable,
            ],
            dropna=False,
        )
        .size()
        .rename(
            "n"
        )
        .reset_index()
    )

    counts[
        "variable"
    ] = variable

    counts = counts.rename(
        columns={
            variable:
            "category"
        }
    )

    categorical_rows.append(
        counts
    )

if categorical_rows:
    preindex_categorical = pd.concat(
        categorical_rows,
        ignore_index=True,
    )

    save_gcs_csv(
        preindex_categorical,
        f"{SENS_DIR}/"
        "selection_bias_preindex_washout_categorical.csv",
    )

display(
    preindex_continuous
)

display(
    source_continuous
)


# 17. Independent Feature-Domain Models


To address whether the nested sequence itself is responsible for the observed pattern,
incremental feature domains are also evaluated independently at the one-year landmark.

The domain definitions are taken directly from the nested feature manifests:

- Intrinsic = M1.
- Laboratory snapshot = M2 − M1.
- Medication = M3 − M2.
- Longitudinal = M4 − M3.
- Social/context = M5 − M4.

M1 and the all-features M5 model use the already-frozen primary predictions. The four
incremental domains are trained independently with the same XGBoost search space and
20-trial optimization used in the primary workflow.


In [ ]:
domain_features = {
    "Intrinsic_M1": (
        model_features[
            "M1"
        ]
    ),
    "Snapshot_only": [
        f
        for f in model_features[
            "M2"
        ]
        if f not in set(
            model_features[
                "M1"
            ]
        )
    ],
    "Medication_only": [
        f
        for f in model_features[
            "M3"
        ]
        if f not in set(
            model_features[
                "M2"
            ]
        )
    ],
    "Longitudinal_only": [
        f
        for f in model_features[
            "M4"
        ]
        if f not in set(
            model_features[
                "M3"
            ]
        )
    ],
    "Social_only": [
        f
        for f in model_features[
            "M5"
        ]
        if f not in set(
            model_features[
                "M4"
            ]
        )
    ],
    "All_features_M5": (
        model_features[
            "M5"
        ]
    ),
}

for domain, features in domain_features.items():
    print(
        f"{domain:<20}: {len(features):,} predictors"
    )

domain_manifest_rows = []

for domain, features in domain_features.items():
    for order, feature in enumerate(
        features
    ):
        domain_manifest_rows.append(
            {
                "domain": domain,
                "feature_order": order,
                "feature": feature,
            }
        )

save_gcs_csv(
    pd.DataFrame(
        domain_manifest_rows
    ),
    f"{SENS_DIR}/"
    "independent_domain_feature_manifest.csv",
)


## Train the four incremental domains independently


In [ ]:
train_primary = df.loc[
    df[
        "split"
    ].eq("train")
].copy()

cal_primary = df.loc[
    df[
        "split"
    ].eq("cal")
].copy()

test_primary = df.loc[
    df[
        "split"
    ].eq("test")
].copy()

y_train_primary = train_primary[
    PRIMARY_Y
].astype(int).to_numpy()

cv = StratifiedKFold(
    n_splits=DOMAIN_CV_FOLDS,
    shuffle=True,
    random_state=SPLIT_SEED,
)


def optimize_domain_xgb(
    features,
):
    X_train = train_primary[
        features
    ].astype(
        np.float32
    )

    def objective(
        trial
    ):
        params = {
            "objective":
                "binary:logistic",
            "eval_metric":
                "auc",
            "tree_method":
                "hist",
            "gamma":
                trial.suggest_float(
                    "gamma",
                    *XGB_BOUNDS[
                        "gamma"
                    ],
                    log=True,
                ),
            "max_depth":
                trial.suggest_int(
                    "max_depth",
                    *XGB_BOUNDS[
                        "max_depth"
                    ],
                ),
            "min_child_weight":
                trial.suggest_int(
                    "min_child_weight",
                    *XGB_BOUNDS[
                        "min_child_weight"
                    ],
                ),
            "subsample":
                trial.suggest_float(
                    "subsample",
                    *XGB_BOUNDS[
                        "subsample"
                    ],
                ),
            "colsample_bytree":
                trial.suggest_float(
                    "colsample_bytree",
                    *XGB_BOUNDS[
                        "colsample_bytree"
                    ],
                ),
            "learning_rate":
                trial.suggest_float(
                    "learning_rate",
                    *XGB_BOUNDS[
                        "learning_rate"
                    ],
                ),
            "n_estimators":
                trial.suggest_int(
                    "n_estimators",
                    *XGB_BOUNDS[
                        "n_estimators"
                    ],
                ),
            "reg_alpha":
                trial.suggest_float(
                    "reg_alpha",
                    *XGB_BOUNDS[
                        "reg_alpha"
                    ],
                    log=True,
                ),
            "reg_lambda":
                trial.suggest_float(
                    "reg_lambda",
                    *XGB_BOUNDS[
                        "reg_lambda"
                    ],
                    log=True,
                ),
            "scale_pos_weight":
                trial.suggest_float(
                    "scale_pos_weight",
                    1.0,
                    2.0,
                ),
            "n_jobs":
                DOMAIN_XGB_N_JOBS,
            "random_state":
                SPLIT_SEED,
        }

        model = xgb.XGBClassifier(
            **params
        )

        scores = cross_val_score(
            model,
            X_train,
            y_train_primary,
            cv=cv,
            scoring="roc_auc",
            n_jobs=DOMAIN_CV_N_JOBS,
        )

        return float(
            scores.mean()
        )

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(
            seed=SPLIT_SEED
        ),
    )

    study.optimize(
        objective,
        n_trials=DOMAIN_N_TRIALS,
    )

    return (
        dict(
            study.best_params
        ),
        float(
            study.best_value
        ),
    )


domain_result_rows = []
domain_prediction_sets = {}

for domain in [
    "Snapshot_only",
    "Medication_only",
    "Longitudinal_only",
    "Social_only",
]:
    features = domain_features[
        domain
    ]

    best_params, best_cv = optimize_domain_xgb(
        features
    )

    fit_params = {
        **best_params,
        "objective":
            "binary:logistic",
        "eval_metric":
            "auc",
        "tree_method":
            "hist",
        "n_jobs":
            XGB_FINAL_N_JOBS,
        "random_state":
            SPLIT_SEED,
    }

    model = xgb.XGBClassifier(
        **fit_params
    )

    X_train = train_primary[
        features
    ].astype(
        np.float32
    )

    X_cal = cal_primary[
        features
    ].astype(
        np.float32
    )

    X_test = test_primary[
        features
    ].astype(
        np.float32
    )

    y_train = train_primary[
        PRIMARY_Y
    ].astype(int).to_numpy()

    y_cal = cal_primary[
        PRIMARY_Y
    ].astype(int).to_numpy()

    y_test = test_primary[
        PRIMARY_Y
    ].astype(int).to_numpy()

    model.fit(
        X_train,
        y_train,
    )

    calibrator = CalibratedClassifierCV(
        model,
        method="isotonic",
        cv="prefit",
    )

    calibrator.fit(
        X_cal,
        y_cal,
    )

    p_cal = calibrator.predict_proba(
        X_cal
    )[:, 1]

    p_test = calibrator.predict_proba(
        X_test
    )[:, 1]

    threshold = choose_threshold_for_specificity(
        y_cal,
        p_cal,
        TARGET_SPECIFICITY,
    )

    cm = confusion_metrics(
        y_test,
        p_test,
        threshold,
    )

    domain_result_rows.append(
        {
            "domain": domain,
            "n_features": len(
                features
            ),
            "best_cv_auroc": best_cv,
            "AUROC": roc_auc_score(
                y_test,
                p_test,
            ),
            "AUPRC": average_precision_score(
                y_test,
                p_test,
            ),
            "Brier": brier_score_loss(
                y_test,
                p_test,
            ),
            "threshold_cal_90spec": threshold,
            **cm,
        }
    )

    pred = pd.concat(
        [
            pd.DataFrame(
                {
                    ID_COL:
                        cal_primary[
                            ID_COL
                        ].astype(int),
                    "split":
                        "cal",
                    "y_true":
                        y_cal,
                    "p_calibrated":
                        p_cal,
                }
            ),
            pd.DataFrame(
                {
                    ID_COL:
                        test_primary[
                            ID_COL
                        ].astype(int),
                    "split":
                        "test",
                    "y_true":
                        y_test,
                    "p_calibrated":
                        p_test,
                }
            ),
        ],
        ignore_index=True,
    )

    domain_prediction_sets[
        domain
    ] = pred

    save_gcs_csv(
        pred,
        f"{SENS_DIR}/"
        f"{domain}_predictions.csv",
    )

    save_gcs_pickle(
        model,
        f"{SENS_MODEL_DIR}/"
        f"{domain}_xgb.pkl",
    )

    save_gcs_pickle(
        calibrator,
        f"{SENS_MODEL_DIR}/"
        f"{domain}_isotonic.pkl",
    )


## Add the primary M1 and M5 rows


In [ ]:
def evaluate_existing_primary(
    model_id,
    label,
):
    pred = prediction_sets[
        model_id
    ]

    cal = pred.loc[
        pred[
            "split"
        ].eq("cal")
    ]

    test = pred.loc[
        pred[
            "split"
        ].eq("test")
    ]

    y_cal = cal[
        "y_true"
    ].astype(int).to_numpy()

    p_cal = cal[
        "p_calibrated"
    ].to_numpy()

    y_test = test[
        "y_true"
    ].astype(int).to_numpy()

    p_test = test[
        "p_calibrated"
    ].to_numpy()

    threshold = choose_threshold_for_specificity(
        y_cal,
        p_cal,
        TARGET_SPECIFICITY,
    )

    return {
        "domain": label,
        "n_features": len(
            model_features[
                model_id
            ]
        ),
        "best_cv_auroc": np.nan,
        "AUROC": roc_auc_score(
            y_test,
            p_test,
        ),
        "AUPRC": average_precision_score(
            y_test,
            p_test,
        ),
        "Brier": brier_score_loss(
            y_test,
            p_test,
        ),
        "threshold_cal_90spec": threshold,
        **confusion_metrics(
            y_test,
            p_test,
            threshold,
        ),
    }


domain_result_rows.extend(
    [
        evaluate_existing_primary(
            "M1",
            "Intrinsic_M1",
        ),
        evaluate_existing_primary(
            "M5",
            "All_features_M5",
        ),
    ]
)

independent_domain_results_df = (
    pd.DataFrame(
        domain_result_rows
    )
    .sort_values(
        "domain"
    )
    .reset_index(
        drop=True
    )
)

save_gcs_csv(
    independent_domain_results_df,
    f"{SENS_DIR}/"
    "independent_domain_model_results.csv",
)

display(
    independent_domain_results_df
)


# 18. DeLong Comparisons for Independent Domains with Bonferroni Adjustment


In [ ]:
def compute_midrank(x):
    x = np.asarray(x)
    order = np.argsort(x)
    sorted_x = x[order]

    n = len(x)
    midranks = np.zeros(
        n,
        dtype=float,
    )

    i = 0

    while i < n:
        j = i

        while (
            j < n
            and sorted_x[j]
            == sorted_x[i]
        ):
            j += 1

        midrank = (
            0.5
            * (
                i
                + j
                - 1
            )
            + 1
        )

        midranks[
            i:j
        ] = midrank

        i = j

    out = np.empty(
        n,
        dtype=float,
    )

    out[
        order
    ] = midranks

    return out


def fast_delong(
    predictions_sorted_transposed,
    label_1_count,
):
    m = label_1_count

    n = (
        predictions_sorted_transposed.shape[
            1
        ]
        - m
    )

    k = predictions_sorted_transposed.shape[
        0
    ]

    positive = predictions_sorted_transposed[
        :,
        :m,
    ]

    negative = predictions_sorted_transposed[
        :,
        m:,
    ]

    tx = np.empty(
        (
            k,
            m,
        )
    )

    ty = np.empty(
        (
            k,
            n,
        )
    )

    tz = np.empty(
        (
            k,
            m + n,
        )
    )

    for r in range(k):
        tx[
            r,
            :,
        ] = compute_midrank(
            positive[
                r,
                :,
            ]
        )

        ty[
            r,
            :,
        ] = compute_midrank(
            negative[
                r,
                :,
            ]
        )

        tz[
            r,
            :,
        ] = compute_midrank(
            predictions_sorted_transposed[
                r,
                :,
            ]
        )

    aucs = (
        tz[
            :,
            :m,
        ].sum(
            axis=1
        )
        / m
        / n
        - (
            m + 1.0
        )
        / (
            2.0
            * n
        )
    )

    v01 = (
        tz[
            :,
            :m,
        ]
        - tx
    ) / n

    v10 = (
        1.0
        - (
            tz[
                :,
                m:,
            ]
            - ty
        )
        / m
    )

    sx = np.atleast_2d(
        np.cov(
            v01
        )
    )

    sy = np.atleast_2d(
        np.cov(
            v10
        )
    )

    covariance = (
        sx / m
        + sy / n
    )

    return aucs, covariance


def delong_roc_test(
    y,
    p1,
    p2,
):
    y = np.asarray(
        y
    ).astype(int)

    p1 = np.asarray(
        p1,
        dtype=float,
    )

    p2 = np.asarray(
        p2,
        dtype=float,
    )

    order = np.argsort(
        -y
    )

    m = int(
        y.sum()
    )

    preds = np.vstack(
        [
            p1,
            p2,
        ]
    )[
        :,
        order,
    ]

    aucs, covariance = fast_delong(
        preds,
        m,
    )

    delta = float(
        aucs[1]
        - aucs[0]
    )

    var_delta = float(
        covariance[
            0,
            0,
        ]
        + covariance[
            1,
            1,
        ]
        - 2
        * covariance[
            0,
            1,
        ]
    )

    se = math.sqrt(
        max(
            var_delta,
            0,
        )
    )

    if se == 0:
        p = (
            1.0
            if delta == 0
            else 0.0
        )

    else:
        z = (
            delta / se
        )

        p = float(
            2
            * stats.norm.sf(
                abs(
                    z
                )
            )
        )

    return {
        "auc_baseline": float(
            aucs[0]
        ),
        "auc_comparator": float(
            aucs[1]
        ),
        "delta_auroc": delta,
        "p_value": p,
    }


m1_test = (
    prediction_sets[
        "M1"
    ]
    .loc[
        prediction_sets[
            "M1"
        ][
            "split"
        ].eq("test")
    ]
    .sort_values(
        ID_COL
    )
    .reset_index(
        drop=True
    )
)

comparison_prediction_sets = {
    **domain_prediction_sets,
    "All_features_M5":
        prediction_sets[
            "M5"
        ],
}

domain_delong_rows = []

for domain, pred in comparison_prediction_sets.items():
    other = (
        pred.loc[
            pred[
                "split"
            ].eq("test")
        ]
        .sort_values(
            ID_COL
        )
        .reset_index(
            drop=True
        )
    )

    if not m1_test[
        [
            ID_COL,
            "y_true",
        ]
    ].equals(
        other[
            [
                ID_COL,
                "y_true",
            ]
        ]
    ):
        raise ValueError(
            f"M1 and {domain} TEST rows differ."
        )

    result = delong_roc_test(
        m1_test[
            "y_true"
        ],
        m1_test[
            "p_calibrated"
        ],
        other[
            "p_calibrated"
        ],
    )

    domain_delong_rows.append(
        {
            "comparison":
                f"Intrinsic_M1 vs {domain}",
            **result,
        }
    )

domain_delong_df = pd.DataFrame(
    domain_delong_rows
)

n_tests = len(
    domain_delong_df
)

domain_delong_df[
    "p_bonferroni"
] = np.minimum(
    1.0,
    domain_delong_df[
        "p_value"
    ]
    * n_tests,
)

save_gcs_csv(
    domain_delong_df,
    f"{SENS_DIR}/"
    "independent_domain_delong_bonferroni.csv",
)

display(
    domain_delong_df
)


# 19. M1 Versus Random Discrimination Baselines


In [ ]:
m1 = m1_test.copy()

y_m1 = m1[
    "y_true"
].astype(int).to_numpy()

p_m1 = m1[
    "p_calibrated"
].to_numpy()

order = np.argsort(
    -y_m1
)

aucs, cov = fast_delong(
    np.vstack(
        [
            p_m1,
        ]
    )[
        :,
        order,
    ],
    int(
        y_m1.sum()
    ),
)

auc_m1 = float(
    aucs[0]
)

var_auc = float(
    np.atleast_2d(
        cov
    )[
        0,
        0,
    ]
)

se_auc = math.sqrt(
    max(
        var_auc,
        0,
    )
)

z_vs_half = (
    (
        auc_m1
        - 0.5
    )
    / se_auc
    if se_auc > 0
    else np.inf
)

p_vs_half = float(
    2
    * stats.norm.sf(
        abs(
            z_vs_half
        )
    )
)

auprc_m1 = average_precision_score(
    y_m1,
    p_m1,
)

prevalence_baseline = float(
    y_m1.mean()
)

random_baseline_df = pd.DataFrame(
    [
        {
            "model": "M1",
            "AUROC": auc_m1,
            "AUROC_random_baseline": 0.5,
            "AUROC_z_vs_0_5": z_vs_half,
            "AUROC_p_vs_0_5": p_vs_half,
            "AUPRC": auprc_m1,
            "AUPRC_random_baseline_prevalence": prevalence_baseline,
        }
    ]
)

save_gcs_csv(
    random_baseline_df,
    f"{SENS_DIR}/"
    "m1_vs_random_baselines.csv",
)

display(
    random_baseline_df
)


# 20. Consolidated Sensitivity Summary


In [ ]:
summary_files = {
    "renal_circularity_audit":
        f"{SENS_DIR}/prelandmark_renal_abnormality_summary.csv",
    "documented_normal_renal":
        f"{SENS_DIR}/documented_normal_prelandmark_renal_results.csv",
    "no_prelandmark_renal_abnormality":
        f"{SENS_DIR}/no_prelandmark_renal_abnormality_results.csv",
    "alternative_outcomes":
        f"{SENS_DIR}/alternative_outcome_model_results.csv",
    "longer_control_followup":
        f"{SENS_DIR}/longer_control_followup_results.csv",
    "fixed_horizon_counts":
        f"{SENS_DIR}/fixed_horizon_cohort_counts.csv",
    "fixed_horizon_results":
        f"{SENS_DIR}/fixed_horizon_model_results.csv",
    "missingness":
        f"{SENS_DIR}/missingness_by_incremental_layer_and_outcome.csv",
    "visit_measurement_density":
        f"{SENS_DIR}/visit_measurement_density_by_outcome.csv",
    "postlandmark_monitoring":
        f"{SENS_DIR}/postlandmark_kidney_monitoring_by_outcome.csv",
    "observation_density_performance":
        f"{SENS_DIR}/performance_by_prelandmark_observation_density.csv",
    "prs_free":
        f"{SENS_DIR}/prs_free_nested_model_results.csv",
    "selection_washout":
        f"{SENS_DIR}/selection_bias_preindex_washout_continuous.csv",
    "selection_1y":
        f"{SENS_DIR}/selection_bias_source_vs_1y_included_continuous.csv",
    "independent_domains":
        f"{SENS_DIR}/independent_domain_model_results.csv",
    "independent_domain_delong":
        f"{SENS_DIR}/independent_domain_delong_bonferroni.csv",
    "m1_random_baseline":
        f"{SENS_DIR}/m1_vs_random_baselines.csv",
}

save_gcs_json(
    {
        "date": FINAL_DATE,
        "landmark": LANDMARK,
        "landmark_days": LANDMARK_DAYS,
        "primary_specificity_policy": TARGET_SPECIFICITY,
        "primary_split_preserved": True,
        "sensitivity_refit_hyperparameters":
            "primary selected XGBoost hyperparameters reused",
        "fixed_horizons_years": [
            3,
            5,
        ],
        "death_table_available":
            bool(
                death_available
            ),
        "outputs":
            summary_files,
    },
    f"{SENS_DIR}/"
    "reviewer_sensitivity_analysis_summary.json",
)

print("=" * 100)
print("NOTEBOOK 06 REVIEWER SENSITIVITY ANALYSES COMPLETE")
print("=" * 100)

for name, path in summary_files.items():
    print(
        f"{name:<36} {path}"
    )
